### Colloborative filtering

* A technique used by the recommender systems to suggest items such as movie, music, or products for users with similar behaviors and taste is defined as colloborarive filtering.
* It suggests movies that is not yet viewed to the target users by predicting that similar users who have liked it in past will like it in future. Key aspects include user-based or item-based filtering.
* Use of matrix vectorization such as Singular Value Decomposition (SVD) or Alternating Least Squares (ALS) for colloborative filtering. 

In [5]:
ls

 Volume in drive C is Windows-SSD
 Volume Serial Number is 1283-82A2

 Directory of C:\Users\Pradheesha\Downloads\Netflix recommendation system

09-04-2026  15:22    <DIR>          .
08-04-2026  08:34    <DIR>          ..
09-04-2026  13:02    <DIR>          .ipynb_checkpoints
09-04-2026  11:37             1,713 app.py
08-04-2026  20:19             6,908 Clustering.ipynb
09-04-2026  15:22            92,618 colloborative_filtering.ipynb
08-04-2026  08:50           102,243 Data_Preprocessing.ipynb
09-04-2026  13:02            11,760 lstm.ipynb
09-04-2026  15:18    <DIR>          models
04-04-2026  11:58         1,493,648 movie.csv
05-04-2026  16:06     3,269,638,009 processed_df.csv
04-04-2026  11:59       690,353,377 rating.csv
09-04-2026  15:04    <DIR>          src
04-04-2026  11:58        21,725,514 tag.csv
               9 File(s)  3,983,425,790 bytes
               5 Dir(s)  321,602,306,048 bytes free


In [1]:
import pandas as pd
import numpy as np

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import TruncatedSVD

In [3]:
#loading the dataset
df = pd.read_csv("C:\\Users\\Pradheesha\\Downloads\\Netflix recommendation system\\processed_df.csv")
movies = pd.read_csv("C:\\Users\\Pradheesha\\Downloads\\Netflix recommendation system\\movie.csv")

#converting the feature "rating"
df['rating'] = df['rating'].astype('float32')

#statements to print the shape and first few rows of the processed_df dataset
print(df.shape)
df.head()

(20265625, 30)


,userId,movieId,rating,timestamp,title,genres,tag,tag_missing,user_avg_rating,movie_avg_rating,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,2,3.5,2005-04-02 23:53:47,Jumanji (1995),Adventure|Children|Fantasy,NaN,1,3.742857,3.213844,...,0,0,0,0,0,0,0,0,0,0
1,1,29,3.5,2005-04-02 23:31:16,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi,NaN,1,3.742857,3.955775,...,0,0,0,0,1,0,1,0,0,0
2,1,32,3.5,2005-04-02 23:33:39,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller,NaN,1,3.742857,3.902560,...,0,0,0,0,1,0,1,1,0,0
3,1,47,3.5,2005-04-02 23:32:07,Seven (a.k.a. Se7en) (1995),Mystery|Thriller,NaN,1,3.742857,4.056273,...,0,0,0,0,1,0,0,1,0,0
4,1,50,3.5,2005-04-02 23:29:40,"Usual Suspects, The (1995)",Crime|Mystery|Thriller,NaN,1,3.742857,4.334675,...,0,0,0,0,1,0,0,1,0,0


In [4]:
df['tag'].isna().all()

np.False_

In [6]:
df['tag'].isna().sum()
df['tag'].isna().mean() * 100

np.float64(98.06843361603701)

In [7]:
df = df.drop(columns=['tag'])

In [9]:
# Encode IDs
user_ids = df['userId'].astype("category").cat.codes
movie_ids = df['movieId'].astype("category").cat.codes

n_users = user_ids.max() + 1
n_movies = movie_ids.max() + 1

# Mapping dictionaries (VERY IMPORTANT)
user_map = dict(zip(df['userId'], user_ids))
movie_map = dict(zip(df['movieId'], movie_ids))

user_inv_map = dict(zip(user_ids, df['userId']))
movie_inv_map = dict(zip(movie_ids, df['movieId']))

# Sparse matrix
sparse_matrix = csr_matrix(
    (df['rating'], (user_ids, movie_ids)),
    shape=(n_users, n_movies)
)

### KNN (Supervised machine learning) algorithm

In [10]:
#using supervised machine learning technique to train the model for user-based colloboratuve fitering
knn_model = NearestNeighbors(
    metric='cosine',
    algorithm='brute'
)

knn_model.fit(sparse_matrix)

,n_neighbors,5
,radius,1.0
,algorithm,'brute'
,leaf_size,30
,metric,'cosine'
,p,2
,metric_params,None
,n_jobs,None


In [11]:
import pickle
with open('models/knn_model.pkl', 'wb') as f:
    pickle.dump(knn_model, f)

print("KNN model saved")

KNN model saved


### XGBoost supervised machine learning algorithm
XGboost is a supervised machine learning algorithm which is used to 

In [7]:
!pip install xgboost

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)


In [12]:
from xgboost import XGBRegressor
drop_cols = ['userId','movieId','rating','timestamp','title','genres']

feature_cols = [col for col in df.columns if col not in drop_cols]

X = df[feature_cols].astype('float32')
y = df['rating']

xgb_model = XGBRegressor(tree_method='hist', n_estimators=50)
xgb_model.fit(X, y)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [13]:
with open('models/xgb_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

print("XGBoost model saved")

XGBoost model saved


In [14]:
import pickle
with open('models/feature_cols.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print("Feature columns saved")

Feature columns saved


### SVD (Single Value Decomposition)
A linear algebra technique which is used to perform matrix factorization is known as SVD. It is primarily used for dimensionality reduction (PCA), noise reduction, data compression, and recommendation systems by identifying the most important structural components of data. 

In [15]:
#training the model using SVD
svd_model = TruncatedSVD(n_components=50, random_state=42)

user_latent = svd_model.fit_transform(sparse_matrix)
movie_latent = svd_model.components_

In [16]:
with open('models/svd_model.pkl', 'wb') as f:
    pickle.dump(svd_model, f)

print("SVD model saved")

SVD model saved
